# 1. Instalação de Dependências
**Preparação do Ambiente de Ingestão**

> Instalação da biblioteca standalone necessária para escrever tabelas e logs transacionais no Azure sem depender do Spark ABFSS.

**Decisão Arquitetural:** Devido a bloqueios de driver nativos no Databricks Free (`KeyProviderException`), utilizamos o pacote `deltalake` acoplado ao Pandas para gerar a tabela em formato Delta puramente via Python.

In [0]:
%pip install deltalake

# 2. Integração com o Motor Utilitário
**Carga de Variáveis e Funções Compartilhadas**

> Nesta etapa, importamos todas as credenciais do `.env` e as funções de leitura/escrita consolidadas pela Squad.

**Decisão Arquitetural:** O comando `%run` aponta diretamente para a nossa nova pasta `99_utils`. Isso injeta as variáveis de ambiente e as instâncias do Azure Data Lake Service Client diretamente no escopo global deste notebook.

In [0]:
%run ../99_utils/feat_squad2_99_helpers

# 3. Mapeamento da Tabela
**Configuração de Metadados da Ingestão**

> Definição explícita de qual entidade (tabela) este script é responsável por capturar.

**Decisão Arquitetural:** Variabilizar o nome da tabela (`ecommerce_pedidos`) evita *hardcoding* nas chamadas de gravação, diminuindo as chances de erro humano ao salvar a tabela em diretórios trocados.

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp

# Oculta logs detalhados do Azure para limpar a saída do console
logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_pedidos"
BRONZE_PATH = get_delta_path("bronze", TABELA)

inicio = log_inicio(f"feat_squad2_bronze_{TABELA}")
log.info(f"Tabela      : {TABELA}")
log.info(f"Bronze Path : {BRONZE_PATH}")

# 4. Motor de Ingestão e Controle de Estado (Regra 2)
**Varredura, Bypass Temporal e Escrita Delta**

> Este é o núcleo do pipeline. O bloco abaixo atende simultaneamente a múltiplas regras de negócio e restrições de ambiente.

**Decisões Arquiteturais:**
1. **Regra 2 do Contrato de Dados (Anti-Duplicidade):** Utilizamos um arquivo de `checkpoint` em texto no Lake para rastrear os `snapshot_ids` já lidos. O laço `for` processa estritamente dados inéditos.
2. **Resolução de Nanosegundos:** O uso da função encapsulada `ler_parquet` (que utiliza Pandas internamente) anula o erro `[PARQUET_TYPE_ILLEGAL]` gerado pelo Spark ao ler tempos em nanosegundos vindos das Azure Functions.
3. **Auditoria Medalhão:** Injeção das colunas obrigatórias `_snapshot_id`, `bronze_ingested_at` e `bronze_source_file` garantindo a rastreabilidade absoluta da origem de cada pedido.

In [0]:
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour

try:
    # 1. Lista todos os snapshots (pastas de tempo) disponíveis no Lake (raw)
    snapshots = sorted(listar_snapshots())
    
    # 2. Lê o arquivo de controle para saber o que já foi processado (Regra 2)
    processados = ler_checkpoint("bronze", TABELA)
    
    # 3. Filtra apenas os pacotes novos
    novos = [s for s in snapshots if s not in processados]
    log.info(f"{len(novos)} snapshot(s) novo(s) para processar")

    total_linhas = 0

    if not novos:
        log.info("Nenhum snapshot novo para processar!")
    else:
        for snapshot_id in novos:
            log.info(f"Processando: {snapshot_id}")
            
            # Lê o parquet contornando o erro de nanosegundos (usando a função do helper)
            df = ler_parquet(snapshot_id, TABELA)
            
            # Adiciona os metadados de auditoria da Arquitetura Medalhão
            df_bronze = df \
                .withColumn("_snapshot_id", lit(snapshot_id)) \
                .withColumn("bronze_ingested_at", current_timestamp()) \
                .withColumn("bronze_source_file", lit(snapshot_id)) \
                .withColumn("_camada", lit("bronze")) \
                .withColumn("ano", year("bronze_ingested_at")) \
                .withColumn("mes", month("bronze_ingested_at")) \
                .withColumn("dia", dayofmonth("bronze_ingested_at")) \
                .withColumn("hora", hour("bronze_ingested_at"))
            
            # Grava no ADLS em formato Delta na pasta squad2/bronze/ecommerce_pedidos
            sucesso = gravar_delta(df_bronze, "bronze", TABELA, partition_by=["ano", "mes", "dia", "hora"])
            count = df_bronze.count()
            total_linhas += count
            
            # Registra na memória que este pacote já foi processado
            processados.add(snapshot_id)
            
            log.info(f"OK {snapshot_id} -> {count} linhas gravadas")
            
        # Salva o log final de arquivos processados no Lake (Garante a Idempotência)
        salvar_checkpoint("bronze", TABELA, processados)
        log.info(f"Total gravado nesta execução: {total_linhas} linhas")

except Exception as e:
    log.error(f"Erro na ingestão Bronze: {str(e)}")
    raise

# 5. Validação da Ingestão e Inspeção de Metadados
**Garantia de Integridade e Auditoria na Camada Bronze**

> Nesta etapa final, realizamos a leitura da tabela Delta recém-persistida para certificar o sucesso da escrita transacional e avaliar a conformidade dos metadados de governança.

**Decisões Arquiteturais:**
1. **Desacoplamento de Leitura:** Utilizamos a função utilitária `ler_delta("bronze", TABELA)` centralizada no nosso helper. Ela tenta ler os dados diretamente via engine Delta e, caso encontre restrições de ambiente do Databricks Free, ativa automaticamente o fallback via Azure SDK para garantir a entrega do DataFrame.
2. **Auditoria de Schema:** Certificamos visualmente se as colunas nativas do negócio (`id_pedido`, `valor_total`, etc.) convivem de forma estável com os quatro novos campos de controle injetados por nossa Squad.
3. **Métrica de Volume Bruto:** A contagem total de linhas serve como baseline para monitorar o crescimento da base e garantir que nenhum micro-lote foi perdido ou dropado durante o bypass de memória.

In [0]:
from deltalake import DeltaTable

try:
    log.info(f"Iniciando testes de validação para a tabela: {TABELA}")
    
    # 1. Carrega os dados consolidados da tabela Delta na Bronze
    df_validacao = ler_delta("bronze", TABELA)
    
    # 2. Executa a contagem total de registros persistidos no Data Lake
    total_registros = df_validacao.count()
    
    # 3. Extrai os metadados nativos do Delta para atestar o particionamento
    caminho_tabela = get_delta_path("bronze", TABELA)
    storage_opts = get_storage_options()
    dt = DeltaTable(caminho_tabela, storage_options=storage_opts)
    particoes = dt.metadata().partition_columns
    particoes_str = ", ".join(particoes) if particoes else "Nenhuma (Tabela Flat)"
    
    # 4. Impressão do Relatório
    print(f"📊 RELATÓRIO DE VALIDAÇÃO BRONZE — SQUAD 2")
    print(f"{'-'*55}")
    print(f"✅ Status da Tabela   : DISPONÍVEL")
    print(f"📌 Caminho Físico     : {caminho_tabela}")
    print(f"🔢 Total de Registros : {total_registros} linhas acumuladas")
    print(f"🗂️  Particionamento    : {particoes_str}")
    print(f"{'-'*55}\n")
    
    # 5. Exibição do Schema estrutural para conferência de tipos
    print("📋 Esquema Estrutural de Metadados:")
    df_validacao.printSchema()
    
    # 6. Exibição de uma Amostra para auditoria visual dos dados
    print("\n👀 Amostra dos Primeiros Registros Ingeridos:")
    display(df_validacao.limit(10))

except Exception as e:
    log.error(f"Falha crítica na validação da Camada Bronze: {str(e)}")
    raise